In [1]:
import google.protobuf; print(google.protobuf.__version__)

7.34.1


In [2]:
!pip install protobuf
!pip install tokenizer

zsh:1: /Users/harshitbudhraja/Downloads/qwen-align-pipeline/qwen-align-pipeline/venv/bin/pip: bad interpreter: /Users/harshitbudhraja/Downloads/qwen-align-pipeline/venv/bin/python3.10: no such file or directory
zsh:1: /Users/harshitbudhraja/Downloads/qwen-align-pipeline/qwen-align-pipeline/venv/bin/pip: bad interpreter: /Users/harshitbudhraja/Downloads/qwen-align-pipeline/venv/bin/python3.10: no such file or directory


# Qwen Inference Pipeline
This script loads the `ppo_ckpt_1500` model and runs inference.

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = '/Users/harshitbudhraja/Downloads/ppo_ckpt_1500'

# Setting fix_mistral_regex=True per the warning in the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Note: we use dtype instead of torch_dtype to avoid the deprecation warning
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map='auto',
    dtype=torch.bfloat16,
    trust_remote_code=True
)
model.eval()
print('Model loaded successfully!')

/Users/harshitbudhraja/Downloads/qwen-align-pipeline/qwen-align-pipeline/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer you are loading from '/Users/harshitbudhraja/Downloads/ppo_ckpt_1500' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 290/290 [00:03<00:00, 86.51it/s]

Model loaded successfully!


In [6]:
def generate_response(prompt, max_new_tokens=512, temperature=0.7):
    # Match the system prompt used during alignment
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant. Always format your code inside <python> tags and wrap explanations in <info> tags."},
        {"role": "user", "content": prompt}
    ]
    
    # Use the model's chat template
    chat_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Tokenize input
    inputs = tokenizer(chat_prompt, return_tensors='pt').to(model.device)
    
    # Generate sequence
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated response
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# Example Usage:
response = generate_response("Write a simple function to add two numbers in Python")
print("Assistant>\n" + response)

Assistant>
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
